# triangle-barycentric — worked example 1: Batch inside-triangle test on a grid of points

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `triangle-barycentric`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In barycentric coordinates, a point `P = A + u*(B-A) + v*(C-A)` lies inside triangle `ABC` if and only if `u >= 0`, `v >= 0`, and `u + v <= 1`. This test applies element-wise to any batch of `(u, v)` pairs, making it trivially vectorizable with PyTorch boolean operators.

## Worked solution

**Step 1 – Generate a grid of (u, v) pairs.** We use `t.meshgrid` over `[0, 1]` to create a dense grid that covers and extends slightly outside the canonical triangle `(0,0)-(1,0)-(0,1)`.

**Step 2 – Flatten to (N, 2).** `t.stack([u_grid.flatten(), v_grid.flatten()], dim=1)` gives us the `(N, 2)` input expected by the inside test.

**Step 3 – Apply the three-predicate test.** We extract `u = uvs[:, 0]` and `v = uvs[:, 1]`, then compute `(u >= 0) & (v >= 0) & (u + v <= 1)`. PyTorch broadcasts `&` element-wise over 1D boolean tensors.

**Step 4 – Count inside points.** `inside.sum().item()` gives the count. For our grid this should be a predictable fraction of the total grid points — roughly half the unit square lies inside the standard simplex.

In [ ]:
import torch as t

def worked1_grid_inside_test(n=11):
    """
    Test which grid points in [0,1]^2 lie inside the canonical triangle.
    Returns (uvs, inside_mask, count).
    """
    t.manual_seed(0)  # not random, but good habit
    us = t.linspace(0, 1, n)
    vs = t.linspace(0, 1, n)
    u_grid, v_grid = t.meshgrid(us, vs, indexing='ij')
    uvs = t.stack([u_grid.flatten(), v_grid.flatten()], dim=1)  # (n*n, 2)

    u = uvs[:, 0]
    v = uvs[:, 1]
    inside = (u >= 0) & (v >= 0) & (u + v <= 1)

    return uvs, inside, inside.sum().item()

uvs, mask, count = worked1_grid_inside_test(n=11)
print(f'total points: {uvs.shape[0]}')
print(f'inside triangle: {int(count)}')
print(f'first inside point: {uvs[mask][0]}')